# 第 14 章: K-means によるクラスタリングの探索と可視化

エルボー法でクラスタ数を選び、ML.NET の SSE と比べて、クラスタごとの特徴を確認する。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: FSharp.Stats, 0.6.0"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter14.KMeans
open MachineLearning.Chapter14.MlNetKMeans
open MachineLearning.Chapter14.Spending

let rows = loadSpending (Path.Combine(dataDir (), "Wholesale.csv"))
let points = toStandardizedPoints SpendingColumns rows

## エルボー法

In [ ]:
let counts = [ 1..10 ]
let mine = sseByClusterCount 10 0 counts points |> List.map snd
let library = counts |> List.map (fun n -> mlNetBestSse n 0 10 points)

[
    Chart.Line(x = counts, y = mine, Name = "自作（初期中心 10 通り）")
    Chart.Line(x = counts, y = library, Name = "ML.NET（k-means++）")
]
|> Chart.combine
|> Chart.withTitle "クラスタ数と SSE"
|> Chart.withXAxisStyle "クラスタ数"
|> Chart.withYAxisStyle "SSE"

In [ ]:
List.zip3 counts mine library
|> List.map (fun (n, a, b) -> {| クラスタ数 = n; 自作 = a; MLNET = b |})
|> List.toArray

## クラスタごとの特徴

In [ ]:
let result = kmeansWithRestarts 10 5 0 points
let summaries = summarizeClusters SpendingColumns result.Labels rows

summaries
|> List.map (fun s ->
    Chart.Column(values = (SpendingColumns |> List.map (fun c -> s.Means[c])), Keys = SpendingColumns, Name = $"クラスタ {s.Cluster}（{s.Count} 件）"))
|> Chart.combine
|> Chart.withTitle "クラスタごとの平均支出額"
|> Chart.withYAxisStyle "平均支出額"